# NutriChat — Main-system comparison on the 200-question development benchmark

This notebook evaluates the fixed **main systems** on:

`data/eval_dataset_200_v2.json`

It runs:

1. Dense RAG without reranking
2. Dense RAG with reranking
3. BM25 without reranking
4. BM25 with reranking
5. Hybrid RRF without reranking
6. Hybrid RRF with reranking
7. LLM-only baseline

This notebook does **not** run chunking, candidate-depth, final-context-size, or other ablation systems.

The comparison configuration is frozen locally:

- chunking: `sentence_15_no_overlap`
- candidate passages: `10`
- final passages: `3`
- retrieval-score threshold: `None`
- reranker: `BAAI/bge-reranker-base`

Metrics in the final benchmark table:

- Pass rate and mean score: all 200 development questions
- Page hit@3 and MRR: the 140 answerable development questions only
- Safety violations and average latency: all 200 development questions


In [1]:
from google.colab import drive, userdata

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NutriChat-RAG/NutriChat"
%cd "$PROJECT_DIR"

!pip install -r requirements.txt
!pip install -e .


Mounted at /content/drive
/content/drive/MyDrive/NutriChat-RAG/NutriChat
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 83.5 MB/s eta 0:00:00
Obtaining file:///content/drive/MyDrive/NutriChat-RAG/NutriChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nutrichat (pyproject.toml) ... done
  Created wheel for nutrichat: filename=nutrichat-0.1.0-0.editable-py3-none-any.whl size=2882 sha256=b27ecfde3330a57533ade639c5b6d38d0c863f0aa4de41200b63aa864b6b5507
  Stored in directory: /tmp/pip-ephem-wheel-cache-uptkyjjs/wheels/13/1f/4c/1f42e06f0c3ace164fb08724e739b140b59d7d29b5214e49d4
Successfully built nutrichat


In [2]:
from __future__ import annotations

import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, TypeVar

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)

from nutrichat.config import (
    ACTIVE_CHUNKING_STRATEGY,
    DEFAULT_MAX_NEW_TOKENS,
    DEFAULT_TEMPERATURE,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
    JUDGE_MODEL,
    RERANKER_MODEL_NAME,
    SystemSpec,
)
from nutrichat.data import load_eval_questions, load_index_artifact
from nutrichat.embeddings import load_embedding_model
from nutrichat.evaluation import run_system_evaluation_incremental
from nutrichat.generation import LLMOnlyPipeline, RAGPipeline
from nutrichat.judging import judge_dataframe_incremental
from nutrichat.reranking import load_reranker
from nutrichat.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRRFRetriever,
)


In [3]:
# ============================================================
# Frozen comparison configuration
# ============================================================

DEV_DATASET_PATH = Path("data/eval_dataset_200_v2.json")

RUN_ID = "dev200_main_systems_c10_f3_nogate_v1"
RUN_ROOT = (
    Path("results")
    / "dev200_main_system_comparison"
    / RUN_ID
)

RAW_DIR = RUN_ROOT / "raw"
JUDGED_DIR = RUN_ROOT / "judged"
SUMMARY_DIR = RUN_ROOT / "summaries"

for directory in [RAW_DIR, JUDGED_DIR, SUMMARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CANDIDATE_K = 10
FINAL_K = 3
RETRIEVAL_SCORE_THRESHOLD = None

RUN_LLM_ONLY = True
DEBUG = False
DEBUG_QUESTION_COUNT = 3

print("Run root:", RUN_ROOT)
print("Dataset:", DEV_DATASET_PATH)
print("Candidate k:", CANDIDATE_K)
print("Final k:", FINAL_K)
print("Retrieval gate:", RETRIEVAL_SCORE_THRESHOLD)


Run root: results/dev200_main_system_comparison/dev200_main_systems_c10_f3_nogate_v1
Dataset: data/eval_dataset_200_v2.json
Candidate k: 10
Final k: 3
Retrieval gate: None


In [4]:
# ============================================================
# Load and validate the development benchmark
# ============================================================

dev_questions = load_eval_questions(str(DEV_DATASET_PATH))

assert len(dev_questions) == 200, (
    f"Expected 200 development questions, found {len(dev_questions)}."
)

question_ids = [str(item["id"]) for item in dev_questions]
assert len(set(question_ids)) == 200, "Development IDs are not unique."

answerable_count = sum(
    item["expected_behavior"] == "answer"
    for item in dev_questions
)

assert answerable_count == 140, (
    f"Expected 140 answerable development questions, found {answerable_count}."
)

questions_to_run = (
    dev_questions[:DEBUG_QUESTION_COUNT]
    if DEBUG
    else dev_questions
)

print("Questions selected:", len(questions_to_run))
print("Answerable questions in full dev set:", answerable_count)


Questions selected: 200
Answerable questions in full dev set: 140


In [5]:
# ============================================================
# Define only the six fixed RAG comparison systems
# ============================================================

MAIN_DEV_SYSTEMS = [
    SystemSpec(
        name="dense_rag_no_reranker_c10_f3_nogate",
        retriever_name="dense",
        use_reranker=False,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
    SystemSpec(
        name="dense_rag_reranker_c10_f3_nogate",
        retriever_name="dense",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
    SystemSpec(
        name="bm25_rag_no_reranker_c10_f3_nogate",
        retriever_name="bm25",
        use_reranker=False,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
    SystemSpec(
        name="bm25_rag_reranker_c10_f3_nogate",
        retriever_name="bm25",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
    SystemSpec(
        name="hybrid_rrf_no_reranker_c10_f3_nogate",
        retriever_name="hybrid_rrf",
        use_reranker=False,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
    SystemSpec(
        name="hybrid_rrf_reranker_c10_f3_nogate",
        retriever_name="hybrid_rrf",
        use_reranker=True,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_K,
        min_retrieval_score=None,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="main_system",
    ),
]

for spec in MAIN_DEV_SYSTEMS:
    print(
        spec.name,
        "| retriever =", spec.retriever_name,
        "| reranker =", spec.use_reranker,
        "| candidate_k =", spec.candidate_k,
        "| final_k =", spec.final_k,
        "| threshold =", spec.min_retrieval_score,
    )


dense_rag_no_reranker_c10_f3_nogate | retriever = dense | reranker = False | candidate_k = 10 | final_k = 3 | threshold = None
dense_rag_reranker_c10_f3_nogate | retriever = dense | reranker = True | candidate_k = 10 | final_k = 3 | threshold = None
bm25_rag_no_reranker_c10_f3_nogate | retriever = bm25 | reranker = False | candidate_k = 10 | final_k = 3 | threshold = None
bm25_rag_reranker_c10_f3_nogate | retriever = bm25 | reranker = True | candidate_k = 10 | final_k = 3 | threshold = None
hybrid_rrf_no_reranker_c10_f3_nogate | retriever = hybrid_rrf | reranker = False | candidate_k = 10 | final_k = 3 | threshold = None
hybrid_rrf_reranker_c10_f3_nogate | retriever = hybrid_rrf | reranker = True | candidate_k = 10 | final_k = 3 | threshold = None


In [6]:
# ============================================================
# Load local retrieval and reranking models
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=userdata.get("NVIDIA_API_KEY"),
    timeout=120.0,
    max_retries=3,
)

index_dir = Path(
    "artifacts"
) / f"index_{ACTIVE_CHUNKING_STRATEGY}"

chunks, embeddings_np = load_index_artifact(index_dir)

embeddings = torch.as_tensor(
    embeddings_np,
    dtype=torch.float32,
    device=DEVICE,
)

embedding_model = load_embedding_model(
    EMBEDDING_MODEL,
    device=DEVICE,
)

reranker_model = load_reranker(
    RERANKER_MODEL_NAME,
    device=DEVICE,
)

assert (
    embeddings.shape[1]
    == embedding_model.get_sentence_embedding_dimension()
), (
    f"Embedding mismatch: saved embeddings are {embeddings.shape[1]}D, "
    f"but {EMBEDDING_MODEL} produces "
    f"{embedding_model.get_sentence_embedding_dimension()}D."
)

print("Chunks:", len(chunks))
print("Embedding shape:", tuple(embeddings.shape))
print("Embedding model:", EMBEDDING_MODEL)
print("Reranker:", RERANKER_MODEL_NAME)
print("Generator:", GENERATION_MODEL)
print("Judge:", JUDGE_MODEL)


Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Chunks: 1212
Embedding shape: (1212, 384)
Embedding model: BAAI/bge-small-en-v1.5
Reranker: BAAI/bge-reranker-base
Generator: nvidia/llama-3.3-nemotron-super-49b-v1
Judge: openai/gpt-oss-120b


/tmp/ipykernel_510/733802784.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  == embedding_model.get_sentence_embedding_dimension()


In [7]:
# ============================================================
# Build retrievers and pipelines
# ============================================================

dense_retriever = DenseRetriever(
    chunks=chunks,
    embeddings=embeddings,
    embedding_model=embedding_model,
)

bm25_retriever = BM25Retriever(
    chunks=chunks,
)

hybrid_retriever = HybridRRFRetriever(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
)

retrievers = {
    "dense": dense_retriever,
    "bm25": bm25_retriever,
    "hybrid_rrf": hybrid_retriever,
}

rag_pipeline = RAGPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
    retrievers=retrievers,
    reranker_model=reranker_model,
)

llm_pipeline = LLMOnlyPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
)


In [8]:
# ============================================================
# Retry helper for remote API failures and rate limits
# ============================================================

T = TypeVar("T")


def run_with_api_backoff(
    operation_name: str,
    operation: Callable[[], T],
    max_retries: int = 12,
) -> T:
    attempt = 0

    while True:
        try:
            return operation()

        except RateLimitError:
            attempt += 1
            if attempt > max_retries:
                raise

            wait_seconds = min(
                900.0,
                60.0 * (2 ** min(attempt - 1, 4)),
            ) + random.uniform(2.0, 12.0)

            print(
                f"{operation_name}: HTTP 429. "
                f"Waiting {wait_seconds:.1f} seconds."
            )
            time.sleep(wait_seconds)

        except (
            APITimeoutError,
            APIConnectionError,
            InternalServerError,
        ) as error:
            attempt += 1
            if attempt > max_retries:
                raise

            wait_seconds = min(
                300.0,
                20.0 * (2 ** min(attempt - 1, 3)),
            ) + random.uniform(1.0, 8.0)

            print(
                f"{operation_name}: {type(error).__name__}. "
                f"Waiting {wait_seconds:.1f} seconds."
            )
            time.sleep(wait_seconds)


## Generate answers for the six main RAG systems

Each system writes to its own incremental CSV. If Colab disconnects, reconnect and rerun this cell. Completed question IDs will be skipped.


In [9]:
for spec in MAIN_DEV_SYSTEMS:
    suffix = "_debug" if DEBUG else ""
    raw_path = RAW_DIR / f"{spec.name}{suffix}.csv"

    print("\n" + "=" * 88)
    print("Running:", spec.name)
    print("=" * 88)

    raw_df = run_with_api_backoff(
        operation_name=f"Raw evaluation: {spec.name}",
        operation=lambda spec=spec, raw_path=raw_path: (
            run_system_evaluation_incremental(
                eval_questions=questions_to_run,
                system_spec=spec,
                pipeline=rag_pipeline,
                output_path=raw_path,
                temperature=spec.temperature,
                max_new_tokens=spec.max_new_tokens,
                is_rag_system=True,
            )
        ),
    )

    expected_rows = len(questions_to_run)

    assert len(raw_df) == expected_rows, (
        f"{spec.name}: expected {expected_rows} rows, found {len(raw_df)}."
    )
    assert raw_df["id"].astype(str).nunique() == expected_rows

    print(spec.name, raw_df.shape)



Running: dense_rag_no_reranker_c10_f3_nogate
Found existing file with 200 completed questions.
Skipping already completed question: A001
Skipping already completed question: A002
Skipping already completed question: A003
Skipping already completed question: A004
Skipping already completed question: A005
Skipping already completed question: A006
Skipping already completed question: A007
Skipping already completed question: A008
Skipping already completed question: A009
Skipping already completed question: A010
Skipping already completed question: A011
Skipping already completed question: A012
Skipping already completed question: A013
Skipping already completed question: A014
Skipping already completed question: A015
Skipping already completed question: A016
Skipping already completed question: A017
Skipping already completed question: A018
Skipping already completed question: A019
Skipping already completed question: A020
Skipping already completed question: A021
Skipping already compl

## Generate the LLM-only baseline

The LLM-only baseline is part of the main comparison table, but it has no retrieval metrics.


In [10]:
if RUN_LLM_ONLY:
    llm_spec = SystemSpec(
        name="llm_only",
        retriever_name="none",
        use_reranker=False,
        candidate_k=0,
        final_k=0,
        min_retrieval_score=None,
        temperature=DEFAULT_TEMPERATURE,
        max_new_tokens=DEFAULT_MAX_NEW_TOKENS,
        experiment_group="dev200_main_system_comparison",
        ablation_factor="rag_vs_llm_only",
    )

    suffix = "_debug" if DEBUG else ""
    llm_raw_path = RAW_DIR / f"llm_only{suffix}.csv"

    llm_raw_df = run_with_api_backoff(
        operation_name="Raw evaluation: LLM-only",
        operation=lambda: run_system_evaluation_incremental(
            eval_questions=questions_to_run,
            system_spec=llm_spec,
            pipeline=llm_pipeline,
            output_path=llm_raw_path,
            temperature=llm_spec.temperature,
            max_new_tokens=llm_spec.max_new_tokens,
            is_rag_system=False,
        ),
    )

    expected_rows = len(questions_to_run)

    assert len(llm_raw_df) == expected_rows
    assert llm_raw_df["id"].astype(str).nunique() == expected_rows

    print("LLM-only:", llm_raw_df.shape)


Found existing file with 49 completed questions.
Skipping already completed question: A001
Skipping already completed question: A002
Skipping already completed question: A003
Skipping already completed question: A004
Skipping already completed question: A005
Skipping already completed question: A006
Skipping already completed question: A007
Skipping already completed question: A008
Skipping already completed question: A009
Skipping already completed question: A010
Skipping already completed question: A011
Skipping already completed question: A012
Skipping already completed question: A013
Skipping already completed question: A014
Skipping already completed question: A015
Skipping already completed question: A016
Skipping already completed question: A017
Skipping already completed question: A018
Skipping already completed question: A019
Skipping already completed question: A020
Skipping already completed question: A021
Skipping already completed question: A022
Skipping already completed 

## Judge every completed system output

Judging is also incremental. Existing valid `(system, id)` judgments are preserved and skipped.


In [14]:
raw_pattern = "*_debug.csv" if DEBUG else "*.csv"
raw_paths = sorted(RAW_DIR.glob(raw_pattern))

assert raw_paths, f"No raw files were found in {RAW_DIR}."

for raw_path in raw_paths:
    raw_df = pd.read_csv(raw_path)

    expected_rows = len(questions_to_run)

    assert len(raw_df) == expected_rows, (
        f"{raw_path.name}: expected {expected_rows} rows, found {len(raw_df)}."
    )
    assert raw_df["id"].astype(str).nunique() == expected_rows

    judged_path = JUDGED_DIR / f"{raw_path.stem}_judged.csv"

    print("\n" + "=" * 88)
    print("Judging:", raw_path.stem)
    print("=" * 88)

    judged_df = run_with_api_backoff(
        operation_name=f"Judging: {raw_path.stem}",
        operation=lambda raw_df=raw_df, judged_path=judged_path: (
            judge_dataframe_incremental(
                eval_df=raw_df,
                client=client,
                judge_model=JUDGE_MODEL,
                output_path=judged_path,
            )
        ),
    )

    assert len(judged_df) == expected_rows, (
        f"{judged_path.name}: expected {expected_rows} judged rows, "
        f"found {len(judged_df)}."
    )
    assert judged_df["id"].astype(str).nunique() == expected_rows
    assert judged_df["pass"].notna().all()

    print(raw_path.stem, judged_df.shape)



Judging: bm25_rag_no_reranker_c10_f3_nogate
Found existing judged file with 200 completed rows.
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A001
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A002
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A003
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A004
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A005
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A006
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A007
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A008
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A009
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A010
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A011
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate::A012
Skipping already judged row: bm25_rag_no_reranker_c10_f3_nogate

## Build the development benchmark results table

Interpretation:

- **Pass rate** and **Mean score** use all 200 development questions.
- **Page hit@3** and **MRR** use only the 140 answerable questions.
- **Safety violations** and **Avg latency** use all 200 questions.
- LLM-only retrieval metrics are displayed as `--`.


In [15]:
DISPLAY_NAMES = {
    "dense_rag_no_reranker_c10_f3_nogate":
        "Dense RAG, no reranker",
    "dense_rag_reranker_c10_f3_nogate":
        "Dense RAG + reranker",
    "bm25_rag_no_reranker_c10_f3_nogate":
        "BM25, no reranker",
    "bm25_rag_reranker_c10_f3_nogate":
        "BM25 + reranker",
    "hybrid_rrf_no_reranker_c10_f3_nogate":
        "Hybrid RRF, no reranker",
    "hybrid_rrf_reranker_c10_f3_nogate":
        "Hybrid RRF + reranker",
    "llm_only":
        "LLM-only",
}


def normalize_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "false": False,
                "0": False,
                "no": False,
            }
        )
    )


judged_pattern = "*_debug_judged.csv" if DEBUG else "*_judged.csv"
judged_paths = sorted(JUDGED_DIR.glob(judged_pattern))

assert judged_paths, f"No judged files were found in {JUDGED_DIR}."

frames = []

for judged_path in judged_paths:
    system_df = pd.read_csv(judged_path)

    expected_rows = len(questions_to_run)

    assert len(system_df) == expected_rows, (
        f"{judged_path.name}: expected {expected_rows} rows, "
        f"found {len(system_df)}."
    )
    assert system_df["id"].astype(str).nunique() == expected_rows
    assert system_df["pass"].notna().all()

    frames.append(system_df)

all_results = pd.concat(
    frames,
    ignore_index=True,
)

all_results["pass_bool"] = normalize_bool(
    all_results["pass"]
)

all_results["safety_violation_bool"] = normalize_bool(
    all_results["safety_violation"]
)

all_results["page_hit_bool"] = normalize_bool(
    all_results["page_hit_at_3"]
)

for column in [
    "overall_score",
    "mrr",
    "latency_seconds",
    "total_seconds",
]:
    if column in all_results.columns:
        all_results[column] = pd.to_numeric(
            all_results[column],
            errors="coerce",
        )


def summarize_system(group: pd.DataFrame) -> pd.Series:
    system_name = str(group["system"].iloc[0])

    answerable_group = group[
        group["expected_behavior"] == "answer"
    ].copy()

    if not DEBUG:
        assert len(group) == 200
        assert len(answerable_group) == 140

    latency_column = (
        "total_seconds"
        if (
            "total_seconds" in group.columns
            and group["total_seconds"].notna().any()
        )
        else "latency_seconds"
    )

    has_retrieval_metrics = (
        answerable_group["page_hit_bool"].notna().any()
    )

    return pd.Series(
        {
            "System": DISPLAY_NAMES.get(
                system_name,
                system_name,
            ),
            "Pass rate": float(
                group["pass_bool"].mean()
            ),
            "Mean score": float(
                group["overall_score"].mean()
            ),
            "Page hit@3": (
                float(answerable_group["page_hit_bool"].mean())
                if has_retrieval_metrics
                else np.nan
            ),
            "MRR": (
                float(answerable_group["mrr"].mean())
                if answerable_group["mrr"].notna().any()
                else np.nan
            ),
            "Safety violations": int(
                group["safety_violation_bool"]
                .fillna(False)
                .sum()
            ),
            "Avg latency": float(
                group[latency_column].mean()
            ),
            "n_all": int(
                group["id"].astype(str).nunique()
            ),
            "n_answerable": int(
                answerable_group["id"].astype(str).nunique()
            ),
            "system_id": system_name,
        }
    )


benchmark_summary = (
    all_results
    .groupby("system", sort=False)
    .apply(summarize_system)
    .reset_index(drop=True)
    .sort_values(
        ["Pass rate", "Mean score"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

benchmark_summary


/tmp/ipykernel_510/2626099443.py:159: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_system)


,System,Pass rate,Mean score,Page hit@3,MRR,Safety violations,Avg latency,n_all,n_answerable,system_id
0,"Hybrid RRF, no reranker",0.975,4.895,0.950000,0.869048,0,23.345679,200,140,hybrid_rrf_no_reranker_c10_f3_nogate
1,Dense RAG + reranker,0.975,4.885,1.000000,0.935714,3,45.275747,200,140,dense_rag_reranker_c10_f3_nogate
2,Hybrid RRF + reranker,0.965,4.870,1.000000,0.935714,2,23.556341,200,140,hybrid_rrf_reranker_c10_f3_nogate
3,BM25 + reranker,0.965,4.835,0.957143,0.896429,1,20.056631,200,140,bm25_rag_reranker_c10_f3_nogate
4,"Dense RAG, no reranker",0.945,4.790,0.921429,0.869048,1,29.838825,200,140,dense_rag_no_reranker_c10_f3_nogate
5,"BM25, no reranker",0.915,4.660,0.864286,0.755952,2,21.974945,200,140,bm25_rag_no_reranker_c10_f3_nogate
6,LLM-only,0.835,4.315,NaN,NaN,1,61.792820,200,140,llm_only


In [16]:
# Display a formatted table similar to the dashboard example.

display_table = benchmark_summary[
    [
        "System",
        "Pass rate",
        "Mean score",
        "Page hit@3",
        "MRR",
        "Safety violations",
        "Avg latency",
    ]
].copy()


def format_percent(value):
    if pd.isna(value):
        return "--"
    return f"{100.0 * value:.1f}%"


def format_decimal(value):
    if pd.isna(value):
        return "--"
    return f"{value:.3f}"


def format_score(value):
    if pd.isna(value):
        return "--"
    return f"{value:.2f}"


def format_latency(value):
    if pd.isna(value):
        return "--"
    return f"{value:.1f}s"


styled_table = (
    display_table.style
    .format(
        {
            "Pass rate": format_percent,
            "Mean score": format_score,
            "Page hit@3": format_percent,
            "MRR": format_decimal,
            "Safety violations": "{:.0f}",
            "Avg latency": format_latency,
        }
    )
    .set_caption(
        "Main-system comparison on the NutriChat development benchmark"
    )
    .hide(axis="index")
    .set_properties(
        subset=["System"],
        **{"font-weight": "600", "text-align": "left"},
    )
    .set_properties(
        subset=[
            "Pass rate",
            "Mean score",
            "Page hit@3",
            "MRR",
            "Safety violations",
            "Avg latency",
        ],
        **{"text-align": "center"},
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "20px"),
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding", "12px 0"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("font-weight", "700"),
                    ("text-align", "center"),
                    ("padding", "10px"),
                    ("border-bottom", "1px solid #d0d7de"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "10px"),
                    ("border-bottom", "1px solid #e5e7eb"),
                ],
            },
        ]
    )
)

display(styled_table)

print(
    "Note: Page hit@3 and MRR are calculated only on "
    "the 140 answerable development questions."
)


System,Pass rate,Mean score,Page hit@3,MRR,Safety violations,Avg latency
"Hybrid RRF, no reranker",97.5%,4.89,95.0%,0.869,0,23.3s
Dense RAG + reranker,97.5%,4.88,100.0%,0.936,3,45.3s
Hybrid RRF + reranker,96.5%,4.87,100.0%,0.936,2,23.6s
BM25 + reranker,96.5%,4.83,95.7%,0.896,1,20.1s
"Dense RAG, no reranker",94.5%,4.79,92.1%,0.869,1,29.8s
"BM25, no reranker",91.5%,4.66,86.4%,0.756,2,22.0s
LLM-only,83.5%,4.32,--,--,1,61.8s


Note: Page hit@3 and MRR are calculated only on the 140 answerable development questions.


In [17]:
# ============================================================
# Save reusable CSV, JSON, and manifest artifacts
# ============================================================

summary_csv_path = (
    SUMMARY_DIR
    / "benchmark_results_dev200.csv"
)

summary_json_path = (
    SUMMARY_DIR
    / "benchmark_results_dev200.json"
)

manifest_path = RUN_ROOT / "run_manifest.json"


benchmark_summary.to_csv(
    summary_csv_path,
    index=False,
)


json_records = (
    benchmark_summary
    .replace({np.nan: None})
    .to_dict(orient="records")
)


artifact = {
    "artifact_version": RUN_ID,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset_path": str(DEV_DATASET_PATH),
    "dataset_role": "development",
    "n_questions": len(questions_to_run),
    "n_answerable_full_dataset": 140,
    "retrieval_metrics_subset": "answerable",
    "candidate_k": CANDIDATE_K,
    "final_k": FINAL_K,
    "retrieval_score_threshold": RETRIEVAL_SCORE_THRESHOLD,
    "systems": json_records,
}

summary_json_path.write_text(
    json.dumps(
        artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


manifest = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": {
        "path": str(DEV_DATASET_PATH),
        "role": "development",
        "sha256": sha256_file(DEV_DATASET_PATH),
        "n_questions": len(dev_questions),
        "n_answerable": answerable_count,
    },
    "configuration": {
        "chunking_strategy": ACTIVE_CHUNKING_STRATEGY,
        "candidate_k": CANDIDATE_K,
        "final_k": FINAL_K,
        "retrieval_score_threshold": None,
        "embedding_model": EMBEDDING_MODEL,
        "reranker_model": RERANKER_MODEL_NAME,
        "generation_model": GENERATION_MODEL,
        "judge_model": JUDGE_MODEL,
    },
    "systems": [
        {
            "name": spec.name,
            "retriever_name": spec.retriever_name,
            "use_reranker": spec.use_reranker,
            "candidate_k": spec.candidate_k,
            "final_k": spec.final_k,
            "min_retrieval_score": spec.min_retrieval_score,
        }
        for spec in MAIN_DEV_SYSTEMS
    ]
    + (
        [
            {
                "name": "llm_only",
                "retriever_name": "none",
                "use_reranker": False,
                "candidate_k": 0,
                "final_k": 0,
                "min_retrieval_score": None,
            }
        ]
        if RUN_LLM_ONLY
        else []
    ),
    "debug": DEBUG,
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Summary CSV:", summary_csv_path)
print("Summary JSON:", summary_json_path)
print("Run manifest:", manifest_path)


Summary CSV: results/dev200_main_system_comparison/dev200_main_systems_c10_f3_nogate_v1/summaries/benchmark_results_dev200.csv
Summary JSON: results/dev200_main_system_comparison/dev200_main_systems_c10_f3_nogate_v1/summaries/benchmark_results_dev200.json
Run manifest: results/dev200_main_system_comparison/dev200_main_systems_c10_f3_nogate_v1/run_manifest.json


## Expected output files

After a complete run:

```text
results/dev200_main_system_comparison/
└── dev200_main_systems_c10_f3_nogate_v1/
    ├── raw/
    │   ├── dense_rag_no_reranker_c10_f3_nogate.csv
    │   ├── dense_rag_reranker_c10_f3_nogate.csv
    │   ├── bm25_rag_no_reranker_c10_f3_nogate.csv
    │   ├── bm25_rag_reranker_c10_f3_nogate.csv
    │   ├── hybrid_rrf_no_reranker_c10_f3_nogate.csv
    │   ├── hybrid_rrf_reranker_c10_f3_nogate.csv
    │   └── llm_only.csv
    ├── judged/
    │   └── *_judged.csv
    ├── summaries/
    │   ├── benchmark_results_dev200.csv
    │   └── benchmark_results_dev200.json
    └── run_manifest.json
```

Do not use this development table as the primary held-out result. Use it to document development-set system comparison. The final test results remain in the dedicated Test-300 notebooks and result folders.
